# SNI-21 VA-DCP setup — clean resumable version

Notebook ini **tidak menjalankan training** dan **tidak mengevaluasi test**. Satu runner menangani ekstraksi, reuse A0, post-audit, object library, A1, dan A2 sehingga state tidak perlu dipindahkan manual antar-cell.

- Smoke saja: CPU cukup.
- Jika full setup akan langsung diikuti training pada sesi yang sama, pilih GPU sebelum mulai agar `/content` tidak hilang akibat pergantian runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        'https://github.com/ediprin/coffee-bean-detection.git', str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)

from coffee_detector.run_sni21_colab_setup import run_sni21_colab_setup
print('SETUP REPO BERHASIL')
print('TRAINING BELUM DIJALANKAN')

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
ADRIAN_ARCHIVE = DRIVE / 'coffee-sni-detection-fullscene-v1/adrian_detection.tar'
FARUQ_ARCHIVE = DRIVE / 'coffee-sni-detection-fullscene-v1/faruq_segmentation.tar'
CROP_ROOT = DRIVE / 'coffee-sni-instance-crop-v1'

assert ADRIAN_ARCHIVE.is_file(), ADRIAN_ARCHIVE
assert FARUQ_ARCHIVE.is_file(), FARUQ_ARCHIVE
assert (CROP_ROOT / 'manifest.csv').is_file(), CROP_ROOT
assert (CROP_ROOT / 'shards').is_dir(), CROP_ROOT / 'shards'

smoke = run_sni21_colab_setup(
    ADRIAN_ARCHIVE,
    FARUQ_ARCHIVE,
    CROP_ROOT,
    '/content',
    profile='smoke',
    seed=42,
)
assert smoke['training_ready'], smoke
print('SMOKE TRAINING_READY =', smoke['training_ready'])
print('TRAINING BELUM DIJALANKAN. TEST TIDAK DIAKSES.')

In [ ]:
from IPython.display import Image as DisplayImage, display

for arm in ('A1', 'A2'):
    print('\n' + arm + ' RAW')
    display(DisplayImage(filename=smoke['arms'][arm]['raw_contact_sheet'], width=900))
    print(arm + ' OVERLAY')
    display(DisplayImage(filename=smoke['arms'][arm]['overlay_contact_sheet'], width=900))

## Full setup — opsional, tetap bukan training

Biarkan `RUN_FULL=False` sampai smoke A1/A2 diperiksa. Object library dan A0 otomatis digunakan ulang.

In [ ]:
RUN_FULL = False

if not RUN_FULL:
    print('FULL SETUP TIDAK DIJALANKAN. TRAINING TIDAK DIJALANKAN.')
else:
    full = run_sni21_colab_setup(
        ADRIAN_ARCHIVE,
        FARUQ_ARCHIVE,
        CROP_ROOT,
        '/content',
        profile='full',
        seed=42,
    )
    assert full['training_ready'], full
    print('FULL TRAINING_READY =', full['training_ready'])
    print('SETUP SELESAI. TRAINING MASIH BELUM DIJALANKAN.')